[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_08/notebook_8_8_llm_differential_diagnosis.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 8.8: LLM-Based Differential Diagnosis System

**Chapter 8: NLP and LLMs - Implementing Journey 7 (Aisha)**

**Journey Connection**: This notebook implements Journey 7 from Chapter 3, where Aisha nearly received the wrong treatment due to an LLM hallucination. For the clinical context and patient story, refer to Chapter 3.

## Learning Objectives

By the end of this notebook, you will be able to:
1. Implement a simulated LLM-based differential diagnosis system
2. Build a knowledge retrieval system (RAG) using medical knowledge
3. Detect hallucinations in LLM outputs
4. Understand failure modes of LLMs in clinical settings
5. Implement safety checks and human-in-the-loop workflows
6. Use production systems like Medley for medical LLM applications

## Clinical Context

LLMs (like GPT-4, Med-PaLM, Claude) show impressive medical knowledge and reasoning. But they can "hallucinate" - generate plausible-sounding but incorrect information.

**Aisha's story**: 42-year-old with fever, cough, chest pain. LLM suggested pulmonary embolism and recommended anticoagulation. A junior resident almost prescribed it. The attending noticed inconsistencies and ordered a chest CT - it was pneumonia, not PE. Anticoagulation would have been dangerous.

**The challenge**:
- LLMs are confident even when wrong
- Hallucinations can be subtle and dangerous
- No built-in mechanism to say "I don't know"
- Human oversight is absolutely essential

**Real-world impact**:
- Studies show 5-15% hallucination rate in medical LLMs
- Up to 30% of LLM suggestions contain at least one clinical inaccuracy
- FDA has not approved any autonomous LLM diagnostic systems
- Human-in-the-loop is regulatory requirement

---

## Setup

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries imported successfully!")
print("\nNote: This notebook simulates LLM behavior for educational purposes.")
print("In production, you would use actual LLM APIs (GPT-4, Claude, Med-PaLM, etc.)")
print("\nProduction Platform: Medley (medley.smile.ki.se) - Medical LLM platform by Karolinska Institutet")

## Part 1: Medical Knowledge Base (for RAG)

We'll create a simplified medical knowledge base. In reality, this would be:
- PubMed articles and clinical guidelines (UpToDate, MDCalc)
- Medical textbooks (Harrison's, Kumar & Clark)
- Clinical databases (OMIM, DynaMed)
- EHR data from previous cases

**RAG (Retrieval-Augmented Generation)**:
- Retrieves relevant knowledge before generating response
- Reduces hallucinations by grounding in facts
- Provides citations and evidence
- Essential for medical applications

In [ ]:
@dataclass
class MedicalCondition:
    """Structured medical knowledge."""
    condition: str
    key_features: List[str]
    diagnostic_tests: List[str]
    treatment: List[str]
    contraindications: List[str]
    prevalence_in_ed: float
    typical_age_range: Tuple[int, int]
    risk_factors: List[str]

# Create medical knowledge base
knowledge_base = [
    MedicalCondition(
        condition='Community-Acquired Pneumonia',
        key_features=['fever', 'cough', 'productive sputum', 'pleuritic chest pain',
                     'dyspnea', 'tachypnea', 'crackles on auscultation'],
        diagnostic_tests=['chest x-ray', 'sputum culture', 'CBC with differential',
                         'blood cultures', 'procalcitonin', 'oxygen saturation'],
        treatment=['empiric antibiotics (amoxicillin-clavulanate or macrolide)',
                  'supportive care', 'oxygen therapy if hypoxic', 'hydration'],
        contraindications=['avoid anticoagulation unless separate indication',
                          'avoid NSAIDs in severe cases'],
        prevalence_in_ed=0.15,  # 15% of respiratory presentations
        typical_age_range=(18, 90),
        risk_factors=['smoking', 'COPD', 'immunosuppression', 'recent viral illness']
    ),
    MedicalCondition(
        condition='Pulmonary Embolism',
        key_features=['sudden onset dyspnea', 'pleuritic chest pain', 'tachycardia',
                     'hemoptysis', 'leg swelling or pain', 'recent immobilization'],
        diagnostic_tests=['d-dimer', 'CT pulmonary angiogram (CTPA)', 'ECG',
                         'Wells score', 'venous doppler ultrasound'],
        treatment=['anticoagulation (LMWH or DOAC)', 'thrombolytics if massive PE',
                  'oxygen therapy', 'hemodynamic support'],
        contraindications=['active bleeding', 'recent surgery or trauma',
                          'hemorrhagic stroke', 'severe thrombocytopenia'],
        prevalence_in_ed=0.02,  # 2% of respiratory presentations
        typical_age_range=(30, 80),
        risk_factors=['prolonged immobility', 'recent surgery', 'malignancy',
                     'oral contraceptives', 'thrombophilia']
    ),
    MedicalCondition(
        condition='Acute Bronchitis',
        key_features=['cough (productive or dry)', 'mild dyspnea',
                     'normal or low-grade fever', 'wheezing', 'normal vital signs'],
        diagnostic_tests=['usually clinical diagnosis', 'chest x-ray only if pneumonia suspected',
                         'pulse oximetry'],
        treatment=['supportive care', 'bronchodilators if wheezing',
                  'cough suppressants', 'avoid antibiotics (usually viral)'],
        contraindications=['avoid antibiotics unless bacterial infection confirmed'],
        prevalence_in_ed=0.25,  # 25% of respiratory presentations
        typical_age_range=(18, 65),
        risk_factors=['smoking', 'air pollution exposure', 'recent viral URTI']
    ),
    MedicalCondition(
        condition='COVID-19 Pneumonia',
        key_features=['fever', 'dry cough', 'dyspnea', 'fatigue',
                     'loss of taste or smell', 'bilateral infiltrates on imaging'],
        diagnostic_tests=['RT-PCR or rapid antigen test', 'chest x-ray or CT',
                         'oxygen saturation', 'inflammatory markers (CRP, D-dimer)'],
        treatment=['supportive care', 'oxygen therapy', 'dexamethasone if severe',
                  'antivirals (remdesivir) if high risk', 'prone positioning'],
        contraindications=['avoid steroids in mild cases', 'avoid NSAIDs initially'],
        prevalence_in_ed=0.10,  # 10% (varies by season and pandemic phase)
        typical_age_range=(18, 90),
        risk_factors=['unvaccinated', 'elderly', 'comorbidities', 'immunocompromised']
    ),
    MedicalCondition(
        condition='Acute Heart Failure Exacerbation',
        key_features=['dyspnea (especially orthopnea and PND)', 'bilateral crackles',
                     'peripheral edema', 'elevated JVP', 'S3 gallop'],
        diagnostic_tests=['BNP or NT-proBNP', 'chest x-ray (pulmonary edema)',
                         'echocardiogram', 'ECG', 'troponin'],
        treatment=['IV diuretics (furosemide)', 'vasodilators (nitroglycerin)',
                  'oxygen therapy', 'BiPAP if severe'],
        contraindications=['avoid excessive diuresis (pre-renal AKI)',
                          'monitor electrolytes closely'],
        prevalence_in_ed=0.08,  # 8% of dyspnea presentations
        typical_age_range=(50, 90),
        risk_factors=['hypertension', 'CAD', 'valvular disease', 'non-adherence to meds']
    ),
    MedicalCondition(
        condition='Pneumothorax',
        key_features=['sudden onset sharp chest pain', 'dyspnea',
                     'reduced breath sounds', 'hyperresonance to percussion'],
        diagnostic_tests=['chest x-ray (upright)', 'CT chest if unclear',
                         'ultrasound (lung sliding)'],
        treatment=['observation if small and stable', 'needle decompression if tension',
                  'chest tube insertion if large or symptomatic'],
        contraindications=['avoid positive pressure ventilation if untreated tension PTX'],
        prevalence_in_ed=0.03,  # 3% of chest pain presentations
        typical_age_range=(15, 40),
        risk_factors=['tall thin males', 'smoking', 'COPD', 'trauma']
    ),
    MedicalCondition(
        condition='Acute Coronary Syndrome',
        key_features=['chest pain (pressure, radiation to arm/jaw)', 'dyspnea',
                     'diaphoresis', 'nausea', 'ECG changes'],
        diagnostic_tests=['ECG (STEMI vs NSTEMI)', 'troponin', 'cardiac biomarkers',
                         'chest x-ray', 'coronary angiography'],
        treatment=['aspirin', 'dual antiplatelet therapy', 'anticoagulation',
                  'PCI if STEMI', 'beta-blockers', 'statins'],
        contraindications=['relative contraindications to anticoagulation',
                          'avoid thrombolytics if high bleeding risk'],
        prevalence_in_ed=0.05,  # 5% of chest pain presentations (many more benign)
        typical_age_range=(40, 90),
        risk_factors=['CAD', 'diabetes', 'hypertension', 'smoking', 'family history']
    )
]

# Convert to DataFrame for easier handling
knowledge_df = pd.DataFrame([
    {
        'condition': kb.condition,
        'prevalence_ed': f"{kb.prevalence_in_ed*100:.0f}%",
        'key_features_count': len(kb.key_features),
        'age_range': f"{kb.typical_age_range[0]}-{kb.typical_age_range[1]}"
    }
    for kb in knowledge_base
])

print("Medical Knowledge Base Initialized")
print(f"  Total conditions: {len(knowledge_base)}")
print(f"\nConditions indexed:")
display(knowledge_df)

print("\n💡 In production: Use vector databases (Pinecone, Weaviate) with BioBERT embeddings")
print("   This allows semantic search over millions of medical documents")

## Part 2: Simulated LLM with Hallucination Capability

We'll simulate LLM behavior with:
- Knowledge retrieval (RAG)
- Reasoning chain generation
- **Intentional hallucinations** to demonstrate failure modes

In production, replace this with actual LLM API calls (OpenRouter, HuggingFace, Medley).

In [ ]:
@dataclass
class DiagnosisCandidate:
    """A single diagnosis in the differential."""
    condition: str
    confidence: float
    reasoning: str
    suggested_tests: List[str]
    suggested_treatments: List[str]
    risk_factors_present: List[str]
    is_hallucinated: bool = False
    hallucination_type: Optional[str] = None

class SimulatedMedicalLLM:
    """
    Simulated LLM for educational purposes.

    Real production systems:
    - GPT-4 via OpenRouter
    - Claude 3.5 via Anthropic API
    - Med-PaLM 2 via Google Vertex AI
    - Medley platform (medley.smile.ki.se) - Karolinska Institutet
    """

    def __init__(self, knowledge_base: List[MedicalCondition], hallucination_rate: float = 0.20):
        self.knowledge_base = knowledge_base
        self.hallucination_rate = hallucination_rate

    def retrieve_relevant_conditions(self, symptoms: List[str],
                                     age: int = 50,
                                     top_k: int = 5) -> List[MedicalCondition]:
        """
        RAG Step 1: Retrieve relevant conditions.

        In production:
        - Use vector embeddings (BioBERT, ClinicalBERT)
        - Cosine similarity search
        - Retrieve from millions of documents
        """
        scores = []

        for condition in self.knowledge_base:
            # Calculate relevance score

            # 1. Symptom overlap
            symptom_matches = sum(
                1 for feature in condition.key_features
                if any(symptom.lower() in feature.lower() for symptom in symptoms)
            )
            symptom_score = symptom_matches / max(len(symptoms), 1)

            # 2. Age appropriateness
            age_min, age_max = condition.typical_age_range
            if age_min <= age <= age_max:
                age_score = 1.0
            else:
                # Penalize if outside typical range
                distance = min(abs(age - age_min), abs(age - age_max))
                age_score = max(0, 1 - distance / 20)

            # 3. Base rate (prevalence)
            prevalence_score = np.sqrt(condition.prevalence_in_ed)  # Square root to avoid over-emphasizing

            # Combined score
            total_score = (
                symptom_score * 0.6 +
                age_score * 0.2 +
                prevalence_score * 0.2
            )

            scores.append((condition, total_score, symptom_matches))

        # Sort by score and return top-k
        scores.sort(key=lambda x: x[1], reverse=True)
        return [condition for condition, score, matches in scores[:top_k]]

    def generate_differential(self, patient_data: Dict) -> List[DiagnosisCandidate]:
        """
        Generate differential diagnosis.

        Steps:
        1. Retrieve relevant conditions (RAG)
        2. Generate reasoning for each
        3. Calculate confidence scores
        4. Rank by likelihood
        5. (Sometimes) Add hallucinated diagnosis
        """
        symptoms = patient_data['symptoms']
        age = patient_data.get('age', 50)

        # Step 1: Retrieve relevant conditions
        relevant_conditions = self.retrieve_relevant_conditions(symptoms, age, top_k=4)

        # Step 2: Generate candidates
        candidates = []

        for condition in relevant_conditions:
            # Calculate confidence
            symptom_matches = sum(
                1 for feature in condition.key_features
                if any(symptom.lower() in feature.lower() for symptom in symptoms)
            )

            # Base confidence on symptom overlap
            confidence = min(0.85, (symptom_matches / len(symptoms)) * 0.9 + 0.1)

            # Generate reasoning
            reasoning_parts = []
            reasoning_parts.append(f"Patient presents with {', '.join(symptoms)}.")

            matching_features = [
                feature for feature in condition.key_features
                if any(symptom.lower() in feature.lower() for symptom in symptoms)
            ]
            if matching_features:
                reasoning_parts.append(
                    f"Key features consistent with {condition.condition}: {', '.join(matching_features[:3])}."
                )

            # Check risk factors
            patient_risk_factors = patient_data.get('risk_factors', [])
            matching_risk_factors = [
                rf for rf in condition.risk_factors
                if any(prf.lower() in rf.lower() for prf in patient_risk_factors)
            ]

            if matching_risk_factors:
                reasoning_parts.append(
                    f"Risk factors present: {', '.join(matching_risk_factors)}."
                )

            reasoning = " ".join(reasoning_parts)

            candidates.append(DiagnosisCandidate(
                condition=condition.condition,
                confidence=confidence,
                reasoning=reasoning,
                suggested_tests=condition.diagnostic_tests[:3],
                suggested_treatments=condition.treatment[:2],
                risk_factors_present=matching_risk_factors,
                is_hallucinated=False
            ))

        # Step 3: Maybe add hallucination
        if np.random.rand() < self.hallucination_rate:
            hallucinated_candidate = self._generate_hallucination(symptoms, candidates)
            # Insert at position 2 or 3 (not first, to be more subtle)
            insert_pos = min(len(candidates), np.random.randint(1, 3))
            candidates.insert(insert_pos, hallucinated_candidate)

        # Step 4: Sort by confidence
        candidates.sort(key=lambda x: x.confidence, reverse=True)

        return candidates[:5]  # Return top 5

    def _generate_hallucination(self, symptoms: List[str],
                               existing_candidates: List[DiagnosisCandidate]) -> DiagnosisCandidate:
        """
        Generate a plausible-sounding but incorrect diagnosis.

        Types of hallucinations:
        1. Overdiagnosis of rare/serious condition
        2. Fabricated test results
        3. Inappropriate treatment recommendations
        4. Misapplied clinical rules
        """
        existing_conditions = [c.condition for c in existing_candidates]

        hallucination_templates = [
            {
                'condition': 'Pulmonary Embolism',
                'confidence': 0.78,  # Inappropriately high
                'reasoning': (
                    f"Patient presents with {', '.join(symptoms)}. "
                    "Chest pain and dyspnea are classic for PE. "
                    "Wells score appears elevated (calculated as 6.5 based on presentation). "  # FABRICATED
                    "Tachycardia strongly suggests PE. "
                    "Immediate anticoagulation warranted."
                ),
                'suggested_tests': ['d-dimer', 'CTPA', 'venous doppler'],
                'suggested_treatments': ['immediate anticoagulation', 'heparin drip', 'close monitoring'],
                'hallucination_type': 'overdiagnosis_serious',
                'danger_level': 'HIGH'  # Anticoagulation for pneumonia is dangerous
            },
            {
                'condition': 'Tuberculous Pleuritis',
                'confidence': 0.65,
                'reasoning': (
                    f"Patient presents with {', '.join(symptoms)}. "
                    "Persistent fever and pleuritic chest pain suggest TB pleuritis. "  # Unlikely without travel history
                    "Night sweats may be subtle in presentation. "  # FABRICATED symptom
                    "Consider TB isolation and treatment."
                ),
                'suggested_tests': ['TB skin test', 'pleural fluid analysis', 'acid-fast bacilli'],
                'suggested_treatments': ['respiratory isolation', 'multi-drug TB therapy'],
                'hallucination_type': 'rare_diagnosis_without_risk_factors',
                'danger_level': 'MEDIUM'
            },
            {
                'condition': 'Acute Pericarditis',
                'confidence': 0.70,
                'reasoning': (
                    f"Patient presents with {', '.join(symptoms)}. "
                    "Pleuritic chest pain is characteristic of pericarditis. "
                    "ECG likely shows diffuse ST elevation (not yet obtained). "  # FABRICATED
                    "Recommend high-dose NSAIDs and colchicine."
                ),
                'suggested_tests': ['ECG', 'echocardiogram', 'troponin'],
                'suggested_treatments': ['high-dose NSAIDs', 'colchicine', 'avoid anticoagulation'],
                'hallucination_type': 'fabricated_test_results',
                'danger_level': 'MEDIUM'
            }
        ]

        # Filter out conditions already in differential
        valid_hallucinations = [
            h for h in hallucination_templates
            if h['condition'] not in existing_conditions
        ]

        if not valid_hallucinations:
            valid_hallucinations = hallucination_templates

        selected = np.random.choice(valid_hallucinations)

        return DiagnosisCandidate(
            condition=selected['condition'],
            confidence=selected['confidence'],
            reasoning=selected['reasoning'],
            suggested_tests=selected['suggested_tests'],
            suggested_treatments=selected['suggested_treatments'],
            risk_factors_present=[],
            is_hallucinated=True,
            hallucination_type=selected['hallucination_type']
        )

# Initialize LLM
medical_llm = SimulatedMedicalLLM(knowledge_base, hallucination_rate=0.20)

print("✓ Simulated Medical LLM initialized")
print(f"  Knowledge base: {len(knowledge_base)} conditions")
print(f"  Hallucination rate: 20% (intentional for demonstration)")
print(f"\n💡 Real LLMs have 5-15% hallucination rate even with RAG")
print(f"   This is why safety checks are essential")

## Part 3: Aisha's Case - Recreating the Dangerous Scenario

Let's recreate what happened to Aisha and see how the LLM behaves.

In [ ]:
# Aisha's presentation
aisha_case = {
    'patient_id': 'Aisha_42F',
    'age': 42,
    'sex': 'Female',
    'symptoms': ['fever', 'cough', 'chest pain', 'dyspnea'],
    'vital_signs': {
        'temperature': 38.5,  # Celsius
        'heart_rate': 105,
        'respiratory_rate': 22,
        'blood_pressure': '125/80',
        'oxygen_saturation': 94  # Room air
    },
    'physical_exam': 'Crackles in right lower lobe on auscultation. No leg swelling or calf tenderness.',
    'risk_factors': ['no recent travel', 'no immobilization', 'no surgery'],
    'true_diagnosis': 'Community-Acquired Pneumonia'  # Ground truth
}

print("="*80)
print("AISHA'S CLINICAL PRESENTATION")
print("="*80)
print(f"\nPatient: {aisha_case['age']}yo {aisha_case['sex']}")
print(f"\nChief Complaint: {', '.join(aisha_case['symptoms'])}")
print(f"\nVital Signs:")
for key, value in aisha_case['vital_signs'].items():
    print(f"  {key.replace('_', ' ').title()}: {value}")
print(f"\nPhysical Exam:\n  {aisha_case['physical_exam']}")
print(f"\nRisk Factors: {', '.join(aisha_case['risk_factors'])}")
print(f"\n[Ground Truth: {aisha_case['true_diagnosis']}]")
print("="*80)

In [ ]:
# Run LLM multiple times to show variability
print("\nGenerating LLM differential diagnosis (3 independent runs)...\n")

for trial in range(3):
    print(f"\n{'='*80}")
    print(f"RUN #{trial + 1}: LLM DIFFERENTIAL DIAGNOSIS")
    print("="*80)

    # Generate differential
    differential = medical_llm.generate_differential(aisha_case)

    # Display results
    for i, diagnosis in enumerate(differential, 1):
        print(f"\n{i}. {diagnosis.condition}")
        print(f"   Confidence: {diagnosis.confidence:.1%}")
        print(f"   Reasoning: {diagnosis.reasoning}")
        print(f"   Suggested Tests: {', '.join(diagnosis.suggested_tests)}")
        print(f"   Suggested Treatments: {', '.join(diagnosis.suggested_treatments)}")

        # Flag hallucinations
        if diagnosis.is_hallucinated:
            print(f"\n   ⚠️  [HALLUCINATION DETECTED]")
            print(f"   Type: {diagnosis.hallucination_type}")
            print(f"   This is an AI-generated error with potentially harmful consequences!")

    # Check if true diagnosis is included
    conditions_suggested = [d.condition for d in differential]
    if aisha_case['true_diagnosis'] in conditions_suggested:
        rank = conditions_suggested.index(aisha_case['true_diagnosis']) + 1
        print(f"\n✓ True diagnosis ({aisha_case['true_diagnosis']}) is #{rank} in differential")
    else:
        print(f"\n❌ True diagnosis ({aisha_case['true_diagnosis']}) NOT in top 5")

    # Check for dangerous hallucinations
    has_dangerous = any(
        d.is_hallucinated and 'anticoagulation' in ' '.join(d.suggested_treatments).lower()
        for d in differential
    )

    if has_dangerous:
        print(f"\n🚨 DANGEROUS: Anticoagulation suggested for likely pneumonia!")
        print(f"   This could cause serious bleeding complications.")

    print("="*80)

print("\n💡 Key Observation:")
print("   - LLM sometimes suggests PE with high confidence")
print("   - Recommends immediate anticoagulation")
print("   - This would be HARMFUL for a pneumonia patient")
print("   - Junior resident might follow the advice")
print("   - Attending physician oversight caught the error in Aisha's real case")

## Part 4: Hallucination Detection System

Can we automatically detect when the LLM is hallucinating?

**Detection methods**:
1. **Knowledge Base Consistency**: Does diagnosis match retrieved knowledge?
2. **Epidemiological Plausibility**: Is the condition likely given base rates?
3. **Clinical Rule Verification**: Are clinical decision rules applied correctly?
4. **Dangerous Recommendation Flagging**: Are there risky treatments suggested?

In [ ]:
@dataclass
class SafetyFlag:
    """Represents a safety concern in LLM output."""
    condition: str
    severity: str  # 'HIGH', 'MEDIUM', 'LOW'
    flag_type: str
    reason: str
    recommendation: str

class HallucinationDetector:
    """
    Detect potential hallucinations in LLM differential diagnoses.

    Methods:
    1. Knowledge base consistency checking
    2. Epidemiological plausibility analysis
    3. Clinical rule verification
    4. Dangerous treatment detection
    """

    def __init__(self, knowledge_base: List[MedicalCondition]):
        self.knowledge_base = knowledge_base
        self.kb_index = {kb.condition: kb for kb in knowledge_base}

    def check_knowledge_consistency(self, diagnosis: DiagnosisCandidate,
                                   patient_symptoms: List[str]) -> Optional[SafetyFlag]:
        """
        Verify diagnosis is consistent with knowledge base.
        """
        # Check if condition is in knowledge base
        kb_condition = self.kb_index.get(diagnosis.condition)

        if kb_condition is None:
            return SafetyFlag(
                condition=diagnosis.condition,
                severity='HIGH',
                flag_type='Unknown Condition',
                reason='Condition not found in validated knowledge base',
                recommendation='Verify diagnosis exists and consult specialist'
            )

        # Check symptom overlap
        kb_features_lower = [f.lower() for f in kb_condition.key_features]
        matching_symptoms = sum(
            1 for symptom in patient_symptoms
            if any(symptom.lower() in feature for feature in kb_features_lower)
        )

        if matching_symptoms < len(patient_symptoms) * 0.4:
            return SafetyFlag(
                condition=diagnosis.condition,
                severity='MEDIUM',
                flag_type='Poor Symptom Match',
                reason=f'Only {matching_symptoms}/{len(patient_symptoms)} symptoms match typical presentation',
                recommendation='Consider alternative diagnoses with better symptom fit'
            )

        return None

    def check_epidemiological_plausibility(self, diagnosis: DiagnosisCandidate,
                                          patient_age: int,
                                          patient_risk_factors: List[str]) -> Optional[SafetyFlag]:
        """
        Check if diagnosis is epidemiologically reasonable.
        """
        kb_condition = self.kb_index.get(diagnosis.condition)
        if kb_condition is None:
            return None

        # Check 1: Age appropriateness
        age_min, age_max = kb_condition.typical_age_range
        if not (age_min <= patient_age <= age_max):
            age_distance = min(abs(patient_age - age_min), abs(patient_age - age_max))
            if age_distance > 15:
                return SafetyFlag(
                    condition=diagnosis.condition,
                    severity='MEDIUM',
                    flag_type='Atypical Age',
                    reason=f'Patient age {patient_age} is outside typical range {age_min}-{age_max}',
                    recommendation='Consider age-appropriate differential diagnoses'
                )

        # Check 2: Rare condition with high confidence
        if kb_condition.prevalence_in_ed < 0.03 and diagnosis.confidence > 0.70:
            # Check if risk factors present
            risk_factor_match = any(
                any(prf.lower() in kbrf.lower() for prf in patient_risk_factors)
                for kbrf in kb_condition.risk_factors
            )

            if not risk_factor_match:
                return SafetyFlag(
                    condition=diagnosis.condition,
                    severity='HIGH',
                    flag_type='Rare Diagnosis Without Risk Factors',
                    reason=f'Rare condition (prevalence {kb_condition.prevalence_in_ed*100:.1f}%) suggested with {diagnosis.confidence:.0%} confidence despite no typical risk factors',
                    recommendation='Rule out common diagnoses first (common things are common)'
                )

        return None

    def check_dangerous_treatments(self, diagnosis: DiagnosisCandidate,
                                  patient_data: Dict) -> Optional[SafetyFlag]:
        """
        Flag potentially dangerous treatment recommendations.
        """
        dangerous_keywords = {
            'anticoagulation': 'bleeding risk',
            'thrombolysis': 'hemorrhage risk',
            'high-dose steroids': 'immunosuppression risk',
            'chemotherapy': 'toxicity risk'
        }

        kb_condition = self.kb_index.get(diagnosis.condition)

        for treatment in diagnosis.suggested_treatments:
            treatment_lower = treatment.lower()

            for dangerous_keyword, risk_type in dangerous_keywords.items():
                if dangerous_keyword in treatment_lower:
                    # Check if contraindications exist
                    if kb_condition and kb_condition.contraindications:
                        return SafetyFlag(
                            condition=diagnosis.condition,
                            severity='HIGH',
                            flag_type='Dangerous Treatment',
                            reason=f'High-risk treatment ({treatment}) suggested. Contraindications: {" ".join(kb_condition.contraindications[:2])}',
                            recommendation='Verify diagnosis before initiating treatment. Check for contraindications.'
                        )

        return None

    def evaluate_differential(self, differential: List[DiagnosisCandidate],
                            patient_data: Dict) -> List[SafetyFlag]:
        """
        Run all detection methods on differential diagnosis.
        """
        flags = []

        symptoms = patient_data['symptoms']
        age = patient_data.get('age', 50)
        risk_factors = patient_data.get('risk_factors', [])

        for diagnosis in differential:
            # Check 1: Knowledge consistency
            flag = self.check_knowledge_consistency(diagnosis, symptoms)
            if flag:
                flags.append(flag)

            # Check 2: Epidemiological plausibility
            flag = self.check_epidemiological_plausibility(diagnosis, age, risk_factors)
            if flag:
                flags.append(flag)

            # Check 3: Dangerous treatments
            flag = self.check_dangerous_treatments(diagnosis, patient_data)
            if flag:
                flags.append(flag)

        return flags

# Initialize detector
detector = HallucinationDetector(knowledge_base)

print("✓ Hallucination Detector initialized")
print("\n  Detection methods:")
print("    1. Knowledge base consistency")
print("    2. Epidemiological plausibility")
print("    3. Dangerous treatment flagging")
print("\n💡 No single method is perfect - we use multiple layers for safety")

In [ ]:
# Test detector on Aisha's case
print("\n" + "="*80)
print("HALLUCINATION DETECTION: Aisha's Case")
print("="*80)

# Generate fresh differential
differential = medical_llm.generate_differential(aisha_case)

print("\nLLM Differential Diagnosis:")
for i, dx in enumerate(differential, 1):
    hallucination_marker = " [HALLUCINATED]" if dx.is_hallucinated else ""
    print(f"  {i}. {dx.condition} ({dx.confidence:.0%}){hallucination_marker}")

# Run detection
safety_flags = detector.evaluate_differential(differential, aisha_case)

if safety_flags:
    print(f"\n⚠️  {len(safety_flags)} SAFETY FLAG(S) RAISED:\n")

    for i, flag in enumerate(safety_flags, 1):
        print(f"{i}. [{flag.severity}] {flag.condition}")
        print(f"   Type: {flag.flag_type}")
        print(f"   Reason: {flag.reason}")
        print(f"   Recommendation: {flag.recommendation}")
        print()

    # Categorize by severity
    high_severity = [f for f in safety_flags if f.severity == 'HIGH']
    if high_severity:
        print(f"\n🚨 {len(high_severity)} HIGH SEVERITY FLAG(S) - REQUIRES IMMEDIATE REVIEW")
else:
    print("\n✓ No safety flags raised")
    print("  (This doesn't guarantee correctness - detector is not perfect)")

print("\n" + "="*80)
print("\n💡 Detection Performance:")
print("   - Can catch obvious hallucinations (rare diagnoses, dangerous treatments)")
print("   - But subtle errors may slip through")
print("   - False positives can cause alert fatigue")
print("   - Human expert review remains essential")

## Part 5: Large-Scale Evaluation

Let's evaluate system performance on multiple clinical cases.

In [ ]:
# Create test dataset
test_cases = [
    {
        'case_id': 'Case_001',
        'age': 67,
        'symptoms': ['fever', 'cough', 'productive sputum', 'dyspnea'],
        'risk_factors': ['COPD', 'smoking'],
        'true_diagnosis': 'Community-Acquired Pneumonia'
    },
    {
        'case_id': 'Case_002',
        'age': 32,
        'symptoms': ['cough', 'mild fever'],
        'risk_factors': ['recent viral illness'],
        'true_diagnosis': 'Acute Bronchitis'
    },
    {
        'case_id': 'Case_003',
        'age': 58,
        'symptoms': ['sudden dyspnea', 'chest pain'],
        'risk_factors': ['recent surgery', 'immobilization'],
        'true_diagnosis': 'Pulmonary Embolism'
    },
    {
        'case_id': 'Case_004',
        'age': 38,
        'symptoms': ['fever', 'cough', 'fatigue'],
        'risk_factors': [],
        'true_diagnosis': 'COVID-19 Pneumonia'
    },
    {
        'case_id': 'Case_005',
        'age': 72,
        'symptoms': ['dyspnea', 'orthopnea', 'leg swelling'],
        'risk_factors': ['hypertension', 'previous MI'],
        'true_diagnosis': 'Acute Heart Failure Exacerbation'
    },
    {
        'case_id': 'Case_006',
        'age': 25,
        'symptoms': ['sudden chest pain', 'dyspnea'],
        'risk_factors': ['tall thin male'],
        'true_diagnosis': 'Pneumothorax'
    },
    {
        'case_id': 'Case_007',
        'age': 61,
        'symptoms': ['chest pain', 'dyspnea', 'diaphoresis'],
        'risk_factors': ['diabetes', 'smoking', 'hypertension'],
        'true_diagnosis': 'Acute Coronary Syndrome'
    },
    {
        'case_id': 'Case_008',
        'age': 45,
        'symptoms': ['fever', 'cough', 'chest pain'],
        'risk_factors': [],
        'true_diagnosis': 'Community-Acquired Pneumonia'
    }
]

# Run evaluation
results = []

print("Running evaluation on 8 test cases...\n")

for case in test_cases:
    # Generate differential
    differential = medical_llm.generate_differential(case)

    # Check if correct diagnosis in differential
    suggested_conditions = [d.condition for d in differential]
    correct_in_diff = case['true_diagnosis'] in suggested_conditions
    rank = suggested_conditions.index(case['true_diagnosis']) + 1 if correct_in_diff else None

    # Run hallucination detection
    safety_flags = detector.evaluate_differential(differential, case)

    # Count hallucinations
    n_hallucinations = sum(1 for d in differential if d.is_hallucinated)
    n_high_severity_flags = sum(1 for f in safety_flags if f.severity == 'HIGH')

    # Check for dangerous recommendations
    has_dangerous = any(
        'dangerous treatment' in f.flag_type.lower()
        for f in safety_flags
    )

    results.append({
        'Case ID': case['case_id'],
        'True Diagnosis': case['true_diagnosis'],
        'In Differential': '✓' if correct_in_diff else '✗',
        'Rank': rank if rank else '-',
        'Hallucinations': n_hallucinations,
        'Safety Flags': len(safety_flags),
        'High Severity': n_high_severity_flags,
        'Dangerous': '⚠️' if has_dangerous else ''
    })

results_df = pd.DataFrame(results)

print("="*80)
print("EVALUATION RESULTS: 8 Clinical Cases")
print("="*80)
display(results_df)

# Calculate metrics
n_cases = len(results_df)
n_correct = sum(1 for r in results if r['In Differential'] == '✓')
avg_rank = np.mean([r['Rank'] for r in results if isinstance(r['Rank'], int)])
n_with_hallucinations = sum(1 for r in results if r['Hallucinations'] > 0)
n_with_dangerous = sum(1 for r in results if r['Dangerous'] == '⚠️')

print(f"\n{'='*80}")
print("PERFORMANCE SUMMARY")
print("="*80)
print(f"\nDiagnostic Accuracy:")
print(f"  Correct diagnosis in differential: {n_correct}/{n_cases} ({n_correct/n_cases*100:.0f}%)")
print(f"  Average rank when correct: {avg_rank:.1f}")

print(f"\nSafety Metrics:")
print(f"  Cases with hallucinations: {n_with_hallucinations}/{n_cases} ({n_with_hallucinations/n_cases*100:.0f}%)")
print(f"  Cases with dangerous recommendations: {n_with_dangerous}/{n_cases} ({n_with_dangerous/n_cases*100:.0f}%)")
print(f"  Total safety flags raised: {sum(r['Safety Flags'] for r in results)}")
print(f"  High severity flags: {sum(r['High Severity'] for r in results)}")

print(f"\n{'='*80}")
print("\n⚠️  KEY INSIGHTS:")
print(f"   - Even with {n_correct/n_cases*100:.0f}% accuracy, hallucinations still occur")
print(f"   - {n_with_dangerous} case(s) had dangerous treatment recommendations")
print(f"   - A single error can cause serious patient harm")
print(f"   - This is why human oversight is mandatory, not optional")

## Part 6: Visualization - Understanding LLM Failures

In [ ]:
# Visualize results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Diagnostic accuracy
ax = axes[0, 0]
accuracy_data = results_df['In Differential'].value_counts()
colors = ['#2ecc71' if label == '✓' else '#e74c3c' for label in accuracy_data.index]
ax.bar(accuracy_data.index, accuracy_data.values, color=colors)
ax.set_title('Diagnostic Accuracy\n(Is true diagnosis in top 5?)', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Cases')
ax.set_xlabel('In Differential')
for i, v in enumerate(accuracy_data.values):
    ax.text(i, v + 0.1, str(v), ha='center', fontsize=11, fontweight='bold')

# 2. Ranking of correct diagnoses
ax = axes[0, 1]
ranks = [r['Rank'] for r in results if isinstance(r['Rank'], int)]
rank_counts = Counter(ranks)
rank_positions = sorted(rank_counts.keys())
rank_values = [rank_counts[r] for r in rank_positions]
ax.bar(rank_positions, rank_values, color='#3498db')
ax.set_title('Rank of Correct Diagnosis\n(When in differential)', fontsize=12, fontweight='bold')
ax.set_xlabel('Rank Position')
ax.set_ylabel('Number of Cases')
ax.set_xticks(rank_positions)
for i, (pos, val) in enumerate(zip(rank_positions, rank_values)):
    ax.text(pos, val + 0.1, str(val), ha='center', fontsize=11, fontweight='bold')

# 3. Safety flags distribution
ax = axes[1, 0]
flag_data = results_df[['Safety Flags', 'High Severity']].sum()
flag_labels = ['Total Flags', 'High Severity']
flag_colors = ['#f39c12', '#e74c3c']
ax.bar(flag_labels, flag_data.values, color=flag_colors)
ax.set_title('Safety Flags Raised', fontsize=12, fontweight='bold')
ax.set_ylabel('Count')
for i, v in enumerate(flag_data.values):
    ax.text(i, v + 0.2, str(v), ha='center', fontsize=11, fontweight='bold')

# 4. Hallucination rate
ax = axes[1, 1]
hallucination_categories = [
    'Cases with\nHallucinations',
    'Cases with\nDangerous Recs'
]
hallucination_counts = [
    n_with_hallucinations,
    n_with_dangerous
]
ax.bar(hallucination_categories, hallucination_counts, color=['#e67e22', '#c0392b'])
ax.set_title('Safety Concerns', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Cases')
ax.axhline(y=n_cases, color='gray', linestyle='--', alpha=0.5, label=f'Total cases ({n_cases})')
for i, v in enumerate(hallucination_counts):
    ax.text(i, v + 0.1, f"{v} ({v/n_cases*100:.0f}%)", ha='center', fontsize=10, fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('llm_differential_diagnosis_evaluation.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nVisualization saved as 'llm_differential_diagnosis_evaluation.png'")

## Part 7: Safe Deployment - Human-in-the-Loop System

How should LLMs be deployed safely in clinical practice?

**Safety principles**:
1. **Automated detection** - Flag potential hallucinations
2. **Mandatory review** - Expert oversight for all outputs
3. **Clear disclaimers** - Transparent about AI limitations
4. **Audit logging** - Track all AI-assisted decisions
5. **Continuous monitoring** - Detect performance degradation

In [ ]:
class SafeClinicalAISystem:
    """
    Production-ready LLM system with comprehensive safety guardrails.
    """

    def __init__(self, llm: SimulatedMedicalLLM,
                 detector: HallucinationDetector,
                 require_attending_review: bool = True):
        self.llm = llm
        self.detector = detector
        self.require_attending_review = require_attending_review
        self.audit_log = []

    def generate_safe_differential(self, patient_data: Dict) -> Dict:
        """
        Generate differential with full safety pipeline.
        """
        # Step 1: Get LLM differential
        differential = self.llm.generate_differential(patient_data)

        # Step 2: Run hallucination detection
        safety_flags = self.detector.evaluate_differential(differential, patient_data)

        # Step 3: Filter out high-risk suggestions
        safe_differential = []
        removed_diagnoses = []

        flagged_conditions = {f.condition for f in safety_flags if f.severity == 'HIGH'}

        for diagnosis in differential:
            if diagnosis.condition in flagged_conditions:
                removed_diagnoses.append(diagnosis)
            else:
                safe_differential.append(diagnosis)

        # Step 4: Determine review requirement
        requires_review = (
            self.require_attending_review or
            len(safety_flags) > 0 or
            len(removed_diagnoses) > 0
        )

        # Step 5: Create audit log entry
        self.audit_log.append({
            'timestamp': pd.Timestamp.now(),
            'patient_id': patient_data.get('patient_id', 'unknown'),
            'differential_count': len(differential),
            'safety_flags': len(safety_flags),
            'removed_count': len(removed_diagnoses),
            'requires_review': requires_review
        })

        return {
            'differential': safe_differential,
            'safety_flags': safety_flags,
            'removed_diagnoses': removed_diagnoses,
            'requires_review': requires_review,
            'review_urgency': 'HIGH' if len(removed_diagnoses) > 0 else 'STANDARD'
        }

    def format_clinical_output(self, result: Dict, patient_data: Dict) -> str:
        """
        Format output for clinical use.
        """
        lines = []
        lines.append("="*80)
        lines.append("AI-ASSISTED DIFFERENTIAL DIAGNOSIS")
        lines.append("="*80)
        lines.append(f"Patient: {patient_data.get('patient_id', 'N/A')}")
        lines.append(f"Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
        lines.append("="*80)
        lines.append("")

        # Differential diagnosis
        lines.append("DIFFERENTIAL DIAGNOSIS:")
        lines.append("")
        for i, dx in enumerate(result['differential'], 1):
            lines.append(f"{i}. {dx.condition} (Confidence: {dx.confidence:.0%})")
            lines.append(f"   Reasoning: {dx.reasoning}")
            lines.append(f"   Suggested Tests: {', '.join(dx.suggested_tests)}")
            lines.append(f"   Suggested Treatments: {', '.join(dx.suggested_treatments)}")
            lines.append("")

        # Safety alerts
        if result['safety_flags']:
            lines.append("\n" + "="*80)
            lines.append("⚠️  SAFETY ALERTS")
            lines.append("="*80)
            for i, flag in enumerate(result['safety_flags'], 1):
                lines.append(f"\n{i}. [{flag.severity}] {flag.condition}")
                lines.append(f"   Type: {flag.flag_type}")
                lines.append(f"   Reason: {flag.reason}")
                lines.append(f"   Recommendation: {flag.recommendation}")

        # Removed diagnoses
        if result['removed_diagnoses']:
            lines.append("\n" + "="*80)
            lines.append("🚫 REMOVED DUE TO HIGH RISK")
            lines.append("="*80)
            for dx in result['removed_diagnoses']:
                lines.append(f"\n- {dx.condition} ({dx.confidence:.0%})")
                lines.append(f"  Reason: Flagged as high-risk by safety system")

        # Review requirement
        if result['requires_review']:
            lines.append("\n" + "="*80)
            lines.append(f"🩺 {result['review_urgency']} PRIORITY: ATTENDING PHYSICIAN REVIEW REQUIRED")
            lines.append("="*80)
            lines.append("This case has been flagged for expert review.")
            lines.append("DO NOT act on AI suggestions without attending physician approval.")

        # Disclaimer
        lines.append("\n" + "="*80)
        lines.append("DISCLAIMER")
        lines.append("="*80)
        lines.append("This is AI-generated assistance only.")
        lines.append("Not a substitute for clinical judgment.")
        lines.append("All recommendations must be verified by licensed physician.")
        lines.append("="*80)

        return "\n".join(lines)

    def get_audit_summary(self) -> pd.DataFrame:
        """
        Get summary of system usage for quality monitoring.
        """
        if not self.audit_log:
            return pd.DataFrame()
        return pd.DataFrame(self.audit_log)

# Create safe system
safe_system = SafeClinicalAISystem(medical_llm, detector, require_attending_review=True)

print("✓ Safe Clinical AI System initialized")
print("\n  Safety features enabled:")
print("    ✓ Hallucination detection")
print("    ✓ High-risk diagnosis filtering")
print("    ✓ Mandatory attending review")
print("    ✓ Comprehensive audit logging")
print("    ✓ Clear disclaimers")

In [ ]:
# Test safe system on Aisha's case
result = safe_system.generate_safe_differential(aisha_case)
clinical_output = safe_system.format_clinical_output(result, aisha_case)

print(clinical_output)

print("\n" + "="*80)
print("WHAT HAPPENED IN AISHA'S REAL CASE?")
print("="*80)
print("\n1. LLM suggested Pulmonary Embolism with anticoagulation")
print("2. Junior resident was about to prescribe heparin")
print("3. Attending physician reviewed the case (human-in-the-loop)")
print("4. Attending noticed:")
print("   - No DVT risk factors present")
print("   - Fever more consistent with infection")
print("   - Physical exam showed crackles (pneumonia)")
print("5. Ordered chest CT → confirmed pneumonia, NOT PE")
print("6. Started appropriate antibiotics")
print("\n✓ Human oversight prevented potentially harmful treatment")
print("\nThis is why FDA requires human review for all AI diagnostic systems.")

## Part 8: Production Deployment with Medley Platform

**Medley** (medley.smile.ki.se) is a medical LLM platform developed by Karolinska Institutet.

### Why use Medley for medical AI?

1. **Medical-specific models**: Fine-tuned on clinical data
2. **Privacy & compliance**: GDPR-compliant, no data retention
3. **Safety features**: Built-in hallucination detection
4. **Audit logging**: Full traceability for clinical use
5. **Regulatory support**: Designed for clinical deployment

### Production example (pseudocode)

In [ ]:
# Production deployment example (pseudocode)
print("="*80)
print("PRODUCTION DEPLOYMENT OPTIONS")
print("="*80)

production_code_medley = '''
# Option 1: Medley Platform (Recommended for clinical use in Sweden)
# Visit: medley.smile.ki.se
# Benefits: Medical-specific, GDPR-compliant, safety features

import requests

MEDLEY_API_URL = "https://medley.smile.ki.se/api/v1/inference"
MEDLEY_API_KEY = "your-api-key-here"  # Obtain from Medley portal

def generate_differential_medley(patient_data):
    prompt = f"""You are a medical AI assistant. Generate a differential diagnosis.

Patient presentation:
- Age: {patient_data['age']}
- Symptoms: {', '.join(patient_data['symptoms'])}
- Risk factors: {', '.join(patient_data['risk_factors'])}

Provide:
1. Top 5 differential diagnoses ranked by likelihood
2. Reasoning for each
3. Suggested diagnostic tests
4. Treatment considerations
5. Red flags or contraindications

Be explicit about uncertainty. If you don't know, say so.
"""

    response = requests.post(
        MEDLEY_API_URL,
        headers={"Authorization": f"Bearer {MEDLEY_API_KEY}"},
        json={
            "model": "medley-clinical-v2",
            "prompt": prompt,
            "max_tokens": 1000,
            "temperature": 0.3,  # Lower temperature for medical use
            "safety_checks": True,  # Enable built-in hallucination detection
            "log_interaction": True  # For audit trail
        }
    )

    return response.json()

# Example usage
result = generate_differential_medley(aisha_case)
print(result['text'])
if result.get('safety_flags'):
    print("⚠️ Safety concerns detected:", result['safety_flags'])
'''

production_code_openrouter = '''
# Option 2: OpenRouter (General-purpose LLMs)
# Visit: https://openrouter.ai/
# Benefits: Multiple models, cost-effective, easy to use

import requests

OPENROUTER_API_URL = "https://openrouter.ai/api/v1/chat/completions"
OPENROUTER_API_KEY = "your-api-key-here"

def generate_differential_openrouter(patient_data):
    response = requests.post(
        OPENROUTER_API_URL,
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "HTTP-Referer": "your-app-url",
            "X-Title": "Medical Differential Diagnosis"
        },
        json={
            "model": "anthropic/claude-3.5-sonnet",  # or "openai/gpt-4"
            "messages": [
                {
                    "role": "system",
                    "content": "You are a medical AI assistant. Be accurate and explicit about uncertainty."
                },
                {
                    "role": "user",
                    "content": f"Generate differential diagnosis for: {patient_data}"
                }
            ],
            "temperature": 0.3
        }
    )
    return response.json()

# Cost: ~$0.003-0.015 per diagnosis with Claude 3.5 Sonnet
'''

production_code_vertex = '''
# Option 3: Google Vertex AI (Med-PaLM 2)
# Visit: https://cloud.google.com/vertex-ai
# Benefits: Medical-specific model, Google Cloud integration

from google.cloud import aiplatform
from vertexai.preview.language_models import TextGenerationModel

# Initialize
aiplatform.init(project="your-project-id", location="us-central1")

def generate_differential_medpalm(patient_data):
    model = TextGenerationModel.from_pretrained("medlm-large")

    prompt = f"""Medical differential diagnosis:
Patient: {patient_data['age']}yo with {', '.join(patient_data['symptoms'])}
Generate ranked differential with reasoning.
"""

    response = model.predict(
        prompt,
        temperature=0.2,
        max_output_tokens=1024
    )

    return response.text
'''

print("\n1. MEDLEY PLATFORM (Recommended for clinical use)")
print("   Platform: medley.smile.ki.se")
print("   Registration: Free for academic research")
print("   Features: Medical models, GDPR compliance, safety checks")
print("\n2. OPENROUTER (General-purpose)")
print("   Platform: https://openrouter.ai/")
print("   Registration: Free tier with $5 credits")
print("   Cost: $0.003-0.015 per diagnosis")
print("\n3. GOOGLE VERTEX AI (Med-PaLM)")
print("   Platform: https://cloud.google.com/vertex-ai")
print("   Registration: $300 free credits")
print("   Features: Medical-specific model")

print("\n" + "="*80)
print("EXAMPLE CODE (copy-paste ready):")
print("="*80)
print("\n# MEDLEY:")
print(production_code_medley)
print("\n# OPENROUTER:")
print(production_code_openrouter)

## Part 9: Key Takeaways

### What We Learned

**1. LLM Capabilities in Medicine**
- Impressive medical knowledge (GPT-4 passes USMLE)
- Can generate plausible differential diagnoses
- Useful for clinical decision support
- **BUT**: Not autonomous - requires human oversight

**2. Hallucinations are Dangerous**
- Not just "wrong answers" - can cause active harm
- Suggestions can be plausible but incorrect
- High confidence doesn't mean correctness
- Examples: Anticoagulation for pneumonia, rare diagnoses without risk factors
- 5-15% hallucination rate even in best models

**3. Hallucination Detection Methods**
- **Knowledge base consistency**: Check against validated sources
- **Epidemiological plausibility**: Consider base rates (common things are common)
- **Clinical rule verification**: Are scoring systems applied correctly?
- **Dangerous treatment flagging**: Identify high-risk recommendations
- **Limitation**: No method is perfect - subtle errors can slip through

**4. Safe Deployment Requirements**
- ✓ Automated safety checks (hallucination detection)
- ✓ High-risk suggestion filtering
- ✓ Mandatory expert review (human-in-the-loop)
- ✓ Clear disclaimers on every output
- ✓ Comprehensive audit logging
- ✓ Continuous performance monitoring
- ✓ Regulatory compliance (FDA, GDPR)

**5. Aisha's Story - Why It Matters**
- LLM suggested PE → anticoagulation (hallucination)
- Junior resident was convinced
- Attending physician caught the error (human-in-the-loop worked)
- Actual diagnosis: Pneumonia (anticoagulation would be harmful)
- **Lesson**: Even one error can cause serious harm - human oversight is essential

### Real-World Context

**Current State of Medical LLMs**:
- GPT-4: 86% on USMLE Step 1, 90% on Step 2
- Med-PaLM 2: Expert-level on medical licensing exams
- But: Evaluation datasets ≠ real clinical practice
- Hallucination rate: 5-15% even with RAG

**Regulatory Status**:
- FDA: No LLM approved for autonomous diagnosis (as of 2025)
- All systems require physician oversight
- Clinical validation studies ongoing
- Liability concerns remain significant

**Best Practices**:
1. Use as **assistant**, not autonomous decision-maker
2. Implement multiple safety layers
3. Require expert review for all outputs
4. Log all interactions for audit
5. Document AI use in medical records
6. Obtain patient informed consent
7. Use medical-specific platforms (Medley) when available

### Connections to Other Chapters

- **Chapter 3 (Seven Journeys)**: Aisha's story provides clinical context
- **Chapter 8 (Other notebooks)**: RAG (8.5), Summarization (8.6), Hallucination Detection (8.7)
- **Chapter 10 (Interpretability)**: Understanding why LLMs generate specific outputs
- **Chapter 11 (Deployment)**: Safety protocols, regulations, human-in-the-loop

---

## Exercises

1. **Prompt engineering**: Modify the system prompt to reduce hallucinations. Does adding "Only suggest diagnoses you are very confident about" help?

2. **Multi-model consensus**: Run the same case through multiple LLMs (GPT-4, Claude, Med-PaLM). Do they agree? Does consensus improve accuracy?

3. **Self-consistency**: Generate 5 responses for the same case. Calculate pairwise agreement. Are confident responses more consistent?

4. **Cost-benefit analysis**: Calculate cost of human review vs cost of missed diagnoses. What's the optimal deployment strategy?

5. **Implement production system**: Use Medley or OpenRouter API to build a real differential diagnosis system with safety checks.

6. **Fairness analysis**: Test on diverse patient populations. Are hallucination rates consistent across demographics?

---

*This notebook is part of "AI in Healthcare" (Volume 1, Chapter 8: NLP and LLMs)*  
*Implements Journey 7 (Aisha - LLM Differential Diagnosis) from Chapter 3*  
*For clinical context, see Chapter 3. For deployment, see Chapter 11.*

---

## 🎉 Congratulations!

You have completed **all 7 clinical journeys** from Chapter 3!

1. ✅ **Marcus** (Sepsis) - False positive challenge (Notebook 4.x)
2. ✅ **Yuki** (AFib) - Prevalence problem (Notebook 5.x)
3. ✅ **Jamal** (Lung nodules) - High FP rate in screening (Notebook 6.x)
4. ✅ **Elena** (Brain tumor) - Human-in-loop success (Notebook 6.x)
5. ✅ **Priya** (Melanoma) - Fairness failure (Notebook 6.x)
6. ✅ **David** (CVD risk) - Calibration vs discrimination (Notebook 5.x)
7. ✅ **Aisha** (LLM) - Hallucination danger (This notebook)

### Key Lessons Across All Journeys

AI in healthcare is not just about accuracy. It requires:
- ✓ Understanding failure modes and edge cases
- ✓ Ensuring fairness across diverse populations
- ✓ Proper calibration for clinical decision-making
- ✓ Human oversight and expert review
- ✓ Safety-first deployment with multiple guardrails
- ✓ Continuous monitoring and quality assurance
- ✓ Humility about limitations and uncertainty

**You are now equipped to build responsible AI systems for healthcare!**

---

## Next Steps

- **Chapter 9**: Deep Learning fundamentals
- **Chapter 10**: Interpretability and explainability
- **Chapter 11**: Clinical deployment and regulatory considerations
- **Notebook 8.9**: Medical transcription with Whisper (optional advanced topic)

---